# 1. The Fundamental Entities

In any RL problem, we divide the universe into two distinct, interacting components:

- **The Agent (The Brain):** The learner and decision-maker. This is your algorithm (and eventually, your neural network). The agent's sole purpose is to map situations to actions in order to maximize a numerical reward signal over time.
- **The Environment (The World):** Everything outside the agent. It is the system the agent interacts with. The environment takes the agent's action, updates its own internal state, and returns a new situation and a reward back to the agent.

# 2. The Interaction Loop (Step-by-Step)

The interaction occurs at discrete time steps, $t = 0, 1, 2, 3, \dots$

Here is exactly how the mathematical loop translates to the code provided in your OpenEnv material:

- **Observation / State ($S_t$):** At time step $t$, the agent receives a representation of the environment's state, $S_t \in \mathcal{S}$ (where $\mathcal{S}$ is the set of all possible states).
    - **In Code:** `observation = environment.observe()`
- **Action ($A_t$):** Based on that state, the agent selects an action, $A_t \in \mathcal{A}(S_t)$ (where $\mathcal{A}(S_t)$ is the set of actions available in state $S_t$).
    - **In Code:** `action = policy.choose(observation)`
- **Reward ($R_{t+1}$):** One time step later, in part as a consequence of its action, the agent receives a numerical reward, $R_{t+1} \in \mathcal{R} \subset \mathbb{R}$.
    - **In Code:** `reward = environment.step(action)`
- **Next State ($S_{t+1}$):** The environment transitions to a new state, and the cycle repeats until a terminal state (like a game over) is reached.
    - **In Code:** `policy.learn(reward)` happens here before the loop restarts.

This continuous cycle generates a sequence or trajectory of data that looks like this:
$$S_0, A_0, R_1, S_1, A_1, R_2, S_2, A_2, R_3, \dots$$

# 3. Dissecting the "Number Guessing Game"

Your OpenEnv tutorial provides a brilliantly simple illustration of this loop in action. Let's break down the variables of that specific environment:

- **The Environment:** The `random.randint(1, 10)` logic holding the target number, and the counter holding `guesses_left = 3`.
- **The Agent:** The `while` loop making guesses.
- **The State ($S_t$):** The number of guesses remaining and the textual feedback from the previous turn ("Warm" or "Cold").
- **The Action ($A_t$):** The integer chosen by the `guess = random.randint(1, 10)` logic.
- **The Reward ($R_t$):** +10 points if the guess exactly matches the target, otherwise 0.

# 4. The Crucial Missing Piece: The Policy ($\pi$)

The tutorial points out a critical flaw in the Number Guessing script: *"But this policy is terrible! It doesn't learn from rewards."*

The agent in that script chooses actions completely at random. In formal RL terminology, the rule an agent uses to decide what to do is called its **Policy**, denoted by $\pi$.

- A **deterministic policy**, $a = \pi(s)$, maps a state directly to a single guaranteed action.
- A **stochastic policy**, $\pi(a|s) = \mathbb{P}[A_t = a | S_t = s]$, gives the probability of taking action $a$ while in state $s$.

The entire goal of Reinforcement Learning is to optimize the policy. We want an agent that observes a "Warm" signal and actively updates its probabilities to guess a closer number next time, rather than just throwing a random integer at the wall.

# Lesson 1.2: The Math of Decision Making (MDPs and Returns)

To mathematically solve the Agent-Environment loop we established in Lesson 1.1, we formalize the entire process using a framework called a **Markov Decision Process (MDP)**.

## 1. The Markov Property

Before defining an MDP, we must understand the "Markov" assumption. A state $S_t$ is said to have the Markov Property if and only if:
$$\mathbb{P}[S_{t+1} | S_t, A_t] = \mathbb{P}[S_{t+1} | S_1, A_1, S_2, A_2, \dots, S_t, A_t]$$

In plain terms: **The future is independent of the past, given the present.** If a state representation contains all necessary information from the history to predict the next state, it is a Markov state.

In the OpenEnv Catch game, the `info_state` provides a flattened 10x5 grid (50 values) showing the exact current position of the ball and the paddle. Because you only need to look at this current frame to know where the ball will fall next (straight down) and where the paddle can move, this environment satisfies the Markov property.

## 2. The MDP Framework

An MDP is formally defined as a tuple of five elements: $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$

*   **State Space ($\mathcal{S}$):** A finite set of all valid states. In the Catch game, this is all possible configurations of the ball and paddle on the 10x5 grid.
*   **Action Space ($\mathcal{A}$):** A finite set of all valid actions the agent can take. In Catch, $\mathcal{A} = \{0, 1, 2\}$, corresponding to Left, Stay, and Right.
*   **Transition Probability Matrix ($\mathcal{P}$):** The dynamics of the environment. It defines the probability of transitioning to state $s'$ given the agent took action $a$ in state $s$:
    $$\mathcal{P}_{ss'}^a = \mathbb{P}[S_{t+1} = s' | S_t = s, A_t = a]$$
*   **Reward Function ($\mathcal{R}$):** The expected immediate reward received after transitioning from state $s$ to $s'$ via action $a$:
    $$\mathcal{R}_s^a = \mathbb{E}[R_{t+1} | S_t = s, A_t = a]$$
*   **Discount Factor ($\gamma$):** A value between $0$ and $1$ ($\gamma \in [0, 1]$) that determines the present value of future rewards.

## 3. Cumulative Return and Discounting

An RL agent does not just want immediate gratification; it wants to maximize the total reward over an entire episode. We define the **Return ($G_t$)** as the total discounted sum of rewards from time step $t$ onward:
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

Why do we need the discount factor ($\gamma$)?

1.  **Mathematical Convergence:** In continuous (non-episodic) tasks, an undiscounted sum of rewards could reach infinity, making it impossible for the agent to compare which policy is better. Discounting bounds the infinite sum.
2.  **Uncertainty:** The future is uncertain. A reward we get tomorrow is worth slightly less than a reward we get today.

If $\gamma = 0$, the agent is perfectly myopic (only cares about $R_{t+1}$). If $\gamma$ approaches $1$, the agent becomes far-sighted.

In the OpenEnv Catch tutorial, an episode is very short (roughly 5 steps) and has a clear success/failure outcome (+1 for catching, 0 for missing). Because the episode naturally terminates, a discount factor of $\gamma = 1$ (no discounting) is mathematically safe and often used in such episodic games.

# Value vs. Policy (Solving the MDP)

Now that we have defined the MDP and the objective (maximizing the Expected Return, $G_t$), how do we actually compute the optimal policy ($\pi^*$)?

There are two primary mathematical quantities we must track, leading to the two dominant branches of Reinforcement Learning algorithms.

## 1. The Mathematical Quantities: Value Functions

To figure out how good a specific state or action is, we define **Value Functions**.

- **State-Value Function ($V_\pi(s)$):** The expected return starting from state $s$, and then strictly following policy $\pi$. It tells the agent "How good is it to be in this situation?"
    $$V_\pi(s) = \mathbb{E}_\pi [G_t | S_t = s]$$
- **Action-Value Function ($Q_\pi(s, a)$):** The expected return starting from state $s$, taking action $a$, and then strictly following policy $\pi$. It tells the agent "How good is it to take this specific action right now?"
    $$Q_\pi(s, a) = \mathbb{E}_\pi [G_t | S_t = s, A_t = a]$$

The relationship between these values and the next state's values is governed by the Bellman Equations, which recursively break down the value function into the immediate reward plus the discounted value of the next state:
$$V_\pi(s) = \sum_{a} \pi(a|s) \sum_{s', r} p(s', r | s, a) [r + \gamma V_\pi(s')]$$

## 2. Approach A: Value-Based Methods (e.g., Q-Learning)

In **Value-Based RL**, we do not explicitly learn a policy function. Instead, we train our agent to perfectly estimate the optimal Action-Value function, $Q^*(s, a)$.

Once the agent knows the true $Q$-values for every action in a given state, the policy becomes trivially simple: just pick the action with the highest Q-value. This is an implicit, deterministic policy:
$$\pi^*(s) = \arg\max_a Q^*(s, a)$$

- **Mechanism:** The agent explores the environment, receives rewards, and uses the Bellman Equation to iteratively update its $Q$-value estimates (using algorithms like Q-Learning or Deep Q-Networks).
- **Connection to OpenEnv:** The `LearningPolicy` class in the provided textbook implements a simulated version of this. It uses an **"Epsilon-greedy exploration"** strategy. Epsilon ($\epsilon$) controls the balance between exploration and exploitation:
    - With probability $\epsilon$, pick a random action from `obs.legal_actions` to discover new values.
    - With probability $1 - \epsilon$, exploit the currently known best action.
    - The code decays epsilon over time (`epsilon = max(0.1, 1.0 - (self.steps / 100))`), slowly shifting from exploration to exploitation as the agent learns the environment.

## 3. Approach B: Policy-Based Methods (e.g., Policy Gradients)

In **Policy-Based RL**, we do not store a value function. Instead, we parameterize the policy itself—usually using a neural network with weights $\theta$. We denote this parameterized policy as $\pi_\theta(a|s)$.

- **Mechanism:** We want to find the parameters $\theta$ that maximize the expected return $J(\theta) = \mathbb{E}_{\pi_\theta}[G_t]$. We do this by calculating the gradient of the objective function and updating the neural network weights via gradient ascent:
    $$\theta_{t+1} = \theta_t + \alpha \nabla_\theta J(\theta)$$
    (Where $\alpha$ is the learning rate). The famous **Policy Gradient Theorem** proves that this gradient can be calculated as: $\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta} [\nabla_\theta \log \pi_\theta(a|s) G_t]$.

### Why use Policy-Based?
- They work exceptionally well in **continuous action spaces** (like controlling the torque of a robot joint), where computing $\arg\max_a$ over infinite actions for Q-Learning is impossible.
- They can naturally learn **true stochastic policies** (e.g., "play Rock 33%, Paper 33%, Scissors 33%"), whereas Value-Based methods generally produce deterministic policies.

## 4. The Modern Hybrid: Actor-Critic

Modern production RL (including algorithms used to train LLMs) often combines both approaches:

- The **Actor** is a policy network $\pi_\theta(a|s)$ that decides what action to take.
- The **Critic** is a value network $V_w(s)$ that evaluates how good the current state is, providing a baseline to tell the Actor if its chosen action was better or worse than expected.

# Epsilon

# The Dilemma of Learning (Exploration vs. Exploitation)

To truly learn an optimal policy, an agent cannot simply choose the action that seems best right now. It must balance what it already knows with what it has yet to discover. This is known as the **Exploration-Exploitation Trade-off**, and it is one of the most fundamental challenges in Reinforcement Learning.

## 1. Defining the Dilemma

When an agent is placed in an environment, its initial Action-Value estimates ($Q$-values) are completely random or initialized to zero. As it takes actions and receives rewards, it faces a constant choice:

-   **Exploitation (Greedy Action):** Choosing the action that yields the highest known expected reward based on current information. You exploit your current knowledge to maximize short-term returns. Mathematically, this is selecting $A_t = \arg\max_a Q(S_t, a)$.
-   **Exploration (Non-Greedy Action):** Choosing a sub-optimal or random action to discover its true value. You sacrifice short-term reward with the hope of gathering information that leads to better long-term returns.

**Why is this a dilemma?** 
- If an agent only **exploits**, it will likely get stuck in a "local optimum" (a strategy that is decent but not the best possible), never realizing that a far superior strategy exists just an action away. 
- If an agent only **explores**, it will behave completely randomly and never accumulate meaningful rewards.

## 2. The $\epsilon$-Greedy (Epsilon-Greedy) Algorithm

The most common and practical mathematical approach to solving this trade-off is the **$\epsilon$-greedy strategy**. We introduce a hyperparameter $\epsilon$ (epsilon), where $0 \leq \epsilon \leq 1$. The policy is defined as:

-   With probability $1 - \epsilon$: The agent **exploits** (chooses the best known action).
-   With probability $\epsilon$: The agent **explores** (chooses an action uniformly at random from the action space, $\mathcal{A}$).

This ensures that every action has a non-zero probability of being selected, guaranteeing that the agent will eventually explore the entire state-action space if given enough time.

## 3. Application in OpenEnv: The LearningPolicy

The OpenEnv tutorial provides a textbook implementation of this exact concept via the `LearningPolicy` class. Let's analyze how it mathematically implements simulated RL.

In the tutorial, the agent is tasked with catching a falling ball. The `LearningPolicy` does not start out knowing how to play perfectly.

**The Code Execution:**
1.  **The Threshold Test:** The algorithm generates a random floating-point number between $0.0$ and $1.0$. It then evaluates if `random.random() < epsilon:`.
2.  **Exploration Branch:** If the random number is less than $\epsilon$, it executes `return random.choice(obs.legal_actions)`. The agent ignores its smart heuristic and just tries moving left, right, or staying entirely at random to see what happens.
3.  **Exploitation Branch:** If the random number is greater than $\epsilon$, it falls to the `else` block: `return self.smart_policy.select_action(obs)`. Here, it exploits the known optimal strategy (moving the paddle toward the ball).

## 4. Epsilon Decay ($\epsilon$-Decay)

A static $\epsilon$ is often inefficient. If $\epsilon = 0.1$, the agent will continue taking random actions 10% of the time even after it has perfectly mastered the environment, needlessly losing rewards.

To fix this, we use **Epsilon Decay**, where the exploration rate is high at the beginning of training and gradually decreases over time. Look precisely at how the OpenEnv tutorial calculates $\epsilon$ at every step:

```python
epsilon = max(0.1, 1.0 - (self.steps / 100))
```

### Mathematical Breakdown of the Decay:
-   **Initial State:** At `self.steps = 0`, $\epsilon = 1.0$. The agent is performing 100% pure exploration, taking completely random actions to map out the environment.
-   **Linear Decay:** As `self.steps` increases, the value of `1.0 - (self.steps / 100)` drops linearly. At step 50, $\epsilon = 0.5$ (50% explore, 50% exploit).
-   **The Floor (Clipping):** The `max(0.1, ...)` function creates an asymptotic floor. Once `self.steps` reaches 90, the equation yields $0.1$. From step 90 to infinity, $\epsilon$ remains locked at $0.1$. This ensures the agent maintains a tiny 10% exploration rate forever, allowing it to adapt if the environment dynamics suddenly change.

## 5. Evaluating the Results

Because of this careful balance, the `LearningPolicy` in your tutorial achieves a highly respectable ~85% success rate over 50 episodes. It fails early on because it is randomly exploring ($\epsilon$ is high), but it successfully catches the ball in the later episodes because it has decayed its $\epsilon$ and is heavily exploiting its learned smart strategy.